In [1]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

In [2]:
DATASET_PATH = "../data/raw/Audio_Speech_Actors_01-24"

In [3]:
def extract_features(file_path):
    
    signal, sr = librosa.load(file_path, sr=None)
    
    # MFCC
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )
    
    # Delta MFCC
    delta = librosa.feature.delta(mfcc)
    
    # Delta-Delta MFCC
    delta2 = librosa.feature.delta(mfcc, order=2)
    
    # Take mean of each feature across time
    mfcc_mean = np.mean(mfcc, axis=1)
    delta_mean = np.mean(delta, axis=1)
    delta2_mean = np.mean(delta2, axis=1)
    
    # Combine all features
    feature_vector = np.concatenate([
        mfcc_mean,
        delta_mean,
        delta2_mean
    ])
    
    return feature_vector

In [4]:
features = []
file_paths = []

for actor in tqdm(os.listdir(DATASET_PATH)):
    
    actor_path = os.path.join(DATASET_PATH, actor)
    
    if not os.path.isdir(actor_path):
        continue
        
    for file in os.listdir(actor_path):
        
        if file.endswith(".wav"):
            
            file_path = os.path.join(actor_path, file)
            
            try:
                feature_vector = extract_features(file_path)
                
                features.append(feature_vector)
                file_paths.append(file)
                
            except:
                print("Error processing:", file)

100%|██████████| 24/24 [01:09<00:00,  2.89s/it]


In [5]:
features = np.array(features)

df_features = pd.DataFrame(features)

df_features["file_name"] = file_paths

df_features.head()

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,file_name
0,-726.217224,68.541420,3.293398,12.205300,5.510278,13.667410,-2.983828,3.098029,-3.310813,-1.564384,...,1.538184e-09,-4.999099e-09,3.076369e-09,-4.614553e-09,6.152737e-09,-1.538184e-09,3.460915e-09,1.922730e-09,-1.922730e-09,03-01-01-01-01-01-01.wav
1,-719.128296,70.201569,1.168397,13.122541,7.836950,14.411290,-4.111360,4.468973,-3.539367,-3.658607,...,3.046883e-09,6.093765e-09,4.570324e-09,1.523441e-09,-3.046883e-09,5.332045e-09,-3.046883e-09,5.332045e-09,-1.142581e-09,03-01-01-01-01-02-01.wav
2,-714.995728,69.689346,3.924564,11.924190,6.421723,11.011614,-2.878103,4.509558,-4.476109,-2.671549,...,-1.416488e-02,-1.359491e-02,-1.295079e-02,-1.223024e-02,-1.143390e-02,-1.057602e-02,-9.686350e-03,-8.801757e-03,-7.952768e-03,03-01-01-01-02-01-01.wav
3,-710.975281,67.564880,5.782240,13.230727,6.190846,12.628252,-1.675169,5.657494,-4.950634,-3.477545,...,-1.245890e-02,-1.138254e-02,-1.104492e-02,-1.258594e-02,-1.402821e-02,-1.450868e-02,-1.359997e-02,-1.002229e-02,-5.964142e-03,03-01-01-01-02-02-01.wav
4,-759.921753,75.783524,6.023605,14.557394,6.454188,14.631508,-3.004551,4.620970,-5.200016,-0.707430,...,-2.241025e-02,-2.091691e-02,-1.987888e-02,-1.897472e-02,-1.749858e-02,-1.506914e-02,-1.225449e-02,-9.423181e-03,-7.121702e-03,03-01-02-01-01-01-01.wav


In [8]:
df_features.shape

(1440, 40)

In [6]:
OUTPUT_PATH = "../data/processed/audio_features.csv"

df_features.to_csv(OUTPUT_PATH, index=False)

print("Features saved successfully!")

Features saved successfully!


## Why 13 Features Are Extracted for Each Metric

### MFCC (Mel-Frequency Cepstral Coefficients)

In this project we extract **MFCC features** from each audio file using the `librosa` library:

```python
librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
```

The parameter `n_mfcc = 13` means that **13 cepstral coefficients** are extracted from the speech signal.

These coefficients capture the **spectral characteristics of the audio signal**, which are important for representing human speech.

---

### Why 13 MFCC Features?

The value **13** is commonly used in speech processing tasks such as:

* Speech recognition
* Emotion recognition
* Speaker identification

This is because the **first 12–13 MFCC coefficients capture the most relevant information** about the speech signal, while higher coefficients tend to represent noise or very fine details.

Thus, using 13 coefficients provides a good balance between:

* capturing meaningful speech information
* keeping the feature space computationally efficient

---

### Interpretation of MFCC Coefficients

Each MFCC coefficient represents different aspects of the **spectral envelope of speech**:

| Coefficient      | Interpretation                  |
| ---------------- | ------------------------------- |
| MFCC_1           | Overall spectral energy         |
| MFCC_2 – MFCC_5  | Major speech formant structures |
| MFCC_6 – MFCC_13 | Finer spectral characteristics  |

These spectral features help differentiate emotions such as **happy, sad, angry, or fearful** in speech.

---

### Delta and Delta-Delta Features

Speech emotion is not only defined by the static spectrum but also by **how the speech signal changes over time**.

To capture these dynamics we compute:

| Feature Type | Description                             | Number of Features |
| ------------ | --------------------------------------- | ------------------ |
| MFCC         | Static spectral features                | 13                 |
| Delta MFCC   | First-order derivative (rate of change) | 13                 |
| Delta² MFCC  | Second-order derivative (acceleration)  | 13                 |

This results in:

```
13 MFCC + 13 Delta + 13 Delta² = 39 total features
```

---

### Why Do We Take the Mean of Features?

MFCC features are computed **for many frames within each audio signal**.

For example:

* Average audio duration ≈ **3.8 seconds**
* Each audio clip may produce **~200 time frames**

The raw MFCC matrix therefore has the shape:

```
13 × number_of_frames
```

To convert this into a **fixed-length feature vector**, we compute the **mean value across time**.

This gives:

```
13 MFCC features per audio file
```

After including Delta and Delta² features, the final feature vector becomes:

```
39 features per audio file
```

---

### Final Feature Representation

Each audio recording is represented by the following features:

```
MFCC_1 … MFCC_13
DELTA_1 … DELTA_13
DELTA2_1 … DELTA2_13
```

Total:

```
39 numerical features per audio file
```

Since the dataset contains **1440 recordings**, the final feature dataset has the shape:

```
1440 × 39
```

These features are later used for **clustering and visualization of emotional patterns in speech**.


## MFCC Feature Extraction Pipeline

MFCC (Mel-Frequency Cepstral Coefficients) are one of the most widely used features in **speech and audio processing**. They transform a raw audio signal into a compact numerical representation that captures the important characteristics of human speech.

The MFCC extraction process consists of several steps.

---

### 1. Audio Signal

The process begins with a **raw audio waveform**.

Each audio file is a time-domain signal sampled at **48 kHz**, meaning the signal contains **48,000 samples per second**.

Speech signals are **non-stationary**, meaning their characteristics change over time.

---

### 2. Framing

To analyze speech effectively, the audio signal is divided into **small overlapping frames**.

Typical frame parameters:

* Frame length: **20–40 ms**
* Frame overlap: **50%**

This allows the signal to be treated as **approximately stationary within each frame**.

---

### 3. Windowing

Each frame is multiplied by a **window function** (usually a Hamming window).

Purpose:

* Reduce **spectral leakage**
* Smooth transitions at frame boundaries

---

### 4. Fourier Transform

Next, a **Fast Fourier Transform (FFT)** is applied to each frame.

This converts the signal from:

```
Time Domain → Frequency Domain
```

The FFT shows how much energy exists at different frequencies.

---

### 5. Mel Filter Bank

Human hearing does not perceive frequencies linearly. Instead, it follows the **Mel scale**, which is more sensitive to lower frequencies.

To simulate human hearing:

* The frequency spectrum is passed through a **bank of triangular filters**.
* These filters are spaced according to the **Mel frequency scale**.

This step produces **Mel-scaled spectral energies**.

---

### 6. Logarithmic Scaling

The energy values are converted to **logarithmic scale**.

Reason:

* Human perception of loudness is **logarithmic rather than linear**.
* Log scaling improves the representation of speech characteristics.

---

### 7. Discrete Cosine Transform (DCT)

Finally, a **Discrete Cosine Transform (DCT)** is applied to the log Mel spectrum.

Purpose:

* Decorrelate the features
* Compress the spectral information

The result of this step is the **MFCC coefficients**.

Typically the first **13 coefficients** are retained because they capture the most significant speech information.

---

### MFCC Pipeline Summary

The complete MFCC extraction process can be summarized as:

```
Audio Signal
      ↓
Framing
      ↓
Windowing
      ↓
Fast Fourier Transform (FFT)
      ↓
Mel Filter Bank
      ↓
Logarithmic Scaling
      ↓
Discrete Cosine Transform (DCT)
      ↓
MFCC Features
```

---

### Why MFCC Is Useful for Emotion Recognition

MFCC features capture important characteristics of speech such as:

* vocal tract shape
* tone and pitch variations
* spectral envelope of speech

These characteristics change when a person expresses different emotions such as:

* happiness
* sadness
* anger
* fear

Therefore, MFCC features provide a **compact and meaningful representation of speech signals**, making them very suitable for **speech emotion analysis and clustering tasks**.
